# Step 1: Data acquisition and cleaning

Downloads MSHA Accident Injuries and Mines datasets, applies documented filtering rules, joins mine context, and writes train/test splits.

**Reference:** `docs/paper_draft.md` Section 6, Step 1; filtering details in `docs/PROGRESS.md`.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data.config import ACCIDENTS_CLEAN_CSV, SUMMARY_JSON, TRAIN_CSV, TEST_CSV
from src.data import ingest

print(f"Project root: {ROOT}")

## Run ingestion

Skips download if raw files already exist in `data/raw/`. Use `force_download=True` to refresh.

In [ ]:
summary = ingest.run(force_download=False)
print(json.dumps(summary, indent=2)[:2000])

In [ ]:
import pandas as pd

with SUMMARY_JSON.open() as f:
    saved = json.load(f)

print("Row counts:")
print(f"  raw accidents: {saved['raw']['accident_rows']:,}")
print(f"  cleaned: {saved['cleaned_rows']:,}")
print(f"  train: {saved['split']['train_rows']:,}")
print(f"  test: {saved['split']['test_rows']:,}")

train = pd.read_csv(TRAIN_CSV, usecols=["DEGREE_INJURY_CD"])
test = pd.read_csv(TEST_CSV, usecols=["DEGREE_INJURY_CD"])
print(f"\nTrain class count: {train['DEGREE_INJURY_CD'].nunique()}")
print(f"Test class count: {test['DEGREE_INJURY_CD'].nunique()}")

## Validation

Run unit tests: `pytest tests/test_data_ingest.py -v`